# Closed-Loop RL

Live mode recomputes reduced contextual-bandit, scaling, remedy, and sequential-maze diagnostics.

### Setup and Dependencies
Imports the trace package, plotting utilities, and configures the default execution mode.


In [ ]:
import os, math, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

RESULT_MODE = "live"
if RESULT_MODE not in {"live", "full_sweep_cache"}:
    raise ValueError("RESULT_MODE must be 'live' or 'full_sweep_cache'")

GREEN, INDIGO, RED, GOLD, GREY, PURPLE, INK = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2", "#b07cc6", "#2b2b2b"
VIR = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(True, color="0.88", lw=0.5)

def _ci_tuple(a):
    a = np.asarray(a, float)
    lo, hi = bootstrap_ci(a)
    return (float(a.mean()), float(a.std()), lo, hi)

def _cache(name):
    print(f"FULL-SWEEP CACHE: {paths.results_dir() / name}")
    return paths.load_result(name)

def _running(rw_2d, w=40):
    rw = np.asarray(rw_2d, float)
    w = max(2, min(w, rw.shape[1] - 1))
    cs = np.cumsum(np.insert(rw, 0, 0.0, axis=1), axis=1)
    rr = (cs[:, w:] - cs[:, :-w]) / w
    return rr.mean(0), np.percentile(rr, 2.5, axis=0), np.percentile(rr, 97.5, axis=0), w

print("RESULT_MODE:", RESULT_MODE)
print("data/results:", paths.results_dir())

from mrl_trace.bandit import run_learning_and_window, train, reward_rate
from mrl_trace.maze import run_sequential

def _series(name, y, min_len=2):
    arr = np.asarray(y, float).ravel()
    if arr.size < min_len or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has insufficient live data for plotting: n={arr.size}")
    return arr

def _values(name, y):
    arr = np.asarray(y, float).ravel()
    if arr.size == 0 or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has no finite live values for plotting")
    return arr

def _smooth(y, win=50):
    arr = _series("curve", y, min_len=2)
    win = int(win)
    if arr.size < max(5, win):
        return arr
    left = win // 2
    right = win - 1 - left
    padded = np.pad(arr, (left, right), mode="edge")
    kernel = np.ones(win, dtype=float) / float(win)
    return np.convolve(padded, kernel, mode="valid")


### Closed-Loop Bandit Learning
Plots the running reward-rate learning curves of the closed-loop bandit compared to chance and trace-ablated controls.


In [ ]:
if RESULT_MODE == "live":
    r = run_learning_and_window(seeds=4, trials=400, delays=(1, 2, 5), tau_leaks=(10.0, 2.0, 0.5))
    src = "LIVE reduced: 4 seeds, 400 trials"
else:
    r = _cache("tier3_results.npy"); src = "full-sweep cache"

delays = np.asarray(r["delays"], float)
fig, ax = plt.subplots(figsize=(4.9, 3.7))
for arr, c, lab in [(r["curve_device"], GREEN, "device trace"), (r["curve_notrace"], GREY, "no trace")]:
    m, lo, hi, w = _running(arr)
    x = np.arange(w, w + len(m))
    ax.plot(x, m, color=c, lw=1.7, label=lab)
    ax.fill_between(x, lo, hi, color=c, alpha=0.2, lw=0)
ax.axhline(0.5, ls="--", color=RED, lw=1.0, label="chance")
ax.set_xlabel("trial"); ax.set_ylabel("reward rate"); ax.set_ylim(0.25, 1.03)
ax.set_title("Closed-loop learning"); ax.legend(frameon=False, fontsize=8); _clean(ax)
for tl, c in zip(sorted(r["reward_rate"].keys(), reverse=True), VIR):
    y = np.asarray(r["reward_rate"][tl], float)
fig.suptitle(f"Bandit delayed-credit result [{src}]", fontsize=11); fig.tight_layout(); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("device final", r["device_final"], "no-trace final", r["notrace_final"])

### State-Space Scaling
Tests the trace model's resilience when scaling the number of states and actions in the environment.


In [ ]:
def _live_scaling():
    out = {}
    for S, A in ((2, 2), (4, 2), (4, 4)):
        chance = 1.0 / A; crit = 0.5 * (1 + chance)
        dev = reward_rate(train(S, A, B=4, D=0.5, trials=400, dt=0.02, tau_leak=4.0), window=120)
        nt = reward_rate(train(S, A, B=4, D=0.5, trials=400, dt=0.02, tau_leak=4.0, no_trace=True), window=120)
        out[(S, A)] = dict(chance=chance, crit=crit, device=_ci_tuple(dev), no_trace=_ci_tuple(nt),
                           trials_to_crit=None, n_converged=int((dev >= crit).sum()), passed=bool(dev.mean() >= crit), n_seeds=4)
    return out
if RESULT_MODE == "live":
    r = _live_scaling(); src = "LIVE oriented: reduced grid"
else:
    r = _cache("tier4_results.npy"); src = "full-sweep cache"

cells_sa = list(r.keys())
labels = [f"{S}x{A}" for S, A in cells_sa]
def _ci_err(entries):
    m = np.array([e[0] for e in entries]); lo = np.array([e[2] for e in entries]); hi = np.array([e[3] for e in entries])
    return m, np.vstack([m - lo, hi - m])
dev, dev_err = _ci_err([r[c]["device"] for c in cells_sa]); nt, nt_err = _ci_err([r[c]["no_trace"] for c in cells_sa])
chance = np.array([r[c]["chance"] for c in cells_sa]); crit = np.array([r[c]["crit"] for c in cells_sa])
x = np.arange(len(cells_sa))
fig, ax = plt.subplots(figsize=(6.4, 3.7))
ax.plot(x, chance, "--", color=RED, lw=1.1, label="chance")
ax.plot(x, crit, ":", color="0.5", lw=1.2, label="criterion")
ax.errorbar(x, dev, yerr=dev_err, marker="o", color=GREEN, capsize=2.5, label="device trace")
ax.errorbar(x, nt, yerr=nt_err, marker="s", color=GREY, capsize=2.5, label="no trace")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_xlabel("state x action size"); ax.set_ylabel("final reward rate")
ax.set_ylim(0, 1.05); ax.set_title(f"Scaling diagnostic [{src}]"); ax.legend(frameon=False, fontsize=8); _clean(ax); plt.show()
print("claim status: live-oriented" if RESULT_MODE == "live" else "claim status: full-sweep cache")

### Algorithm Remedies
Explores exploration bonuses, budget increases, and abstract traces to diagnose limits in the standard model.


In [ ]:
def _live_remedies():
    rows = [
        ("R0 baseline", {}),
        ("R1 more budget", {"trials": 140}),
        ("R2 directed exploration", {"sigma0": 0.45, "sigma1": 0.05}),
        ("R3 abstract trace", {"abstract": True}),
        ("R4 budget + directed", {"trials": 140, "sigma0": 0.45, "sigma1": 0.05}),
    ]
    out = {}; crit = 0.5 * (1 + 1.0 / 4)
    for name, kw0 in rows:
        kw = dict(kw0); trials = kw.pop("trials", 80)
        fr = reward_rate(train(4, 4, B=4, D=0.5, trials=trials, dt=0.02, tau_leak=4.0, **kw), window=min(120, trials))
        out[name] = dict(final=_ci_tuple(fr), passed=bool(fr.mean() >= crit), n_seeds=4)
    return out
if RESULT_MODE == "live":
    r = _live_remedies(); src = "LIVE oriented: reduced 4x4 remedy grid"
else:
    r = _cache("tier5_results.npy"); src = "full-sweep cache"
keys = list(r.keys()); vals = np.array([r[k]["final"][0] for k in keys]); lo = np.array([r[k]["final"][2] for k in keys]); hi = np.array([r[k]["final"][3] for k in keys])
x = np.arange(len(keys))
fig, ax = plt.subplots(figsize=(7.2, 3.7))
ax.bar(x, vals, yerr=np.vstack([vals - lo, hi - vals]), color=GREEN, capsize=3, edgecolor="white")
ax.axhline(0.5 * (1 + 1 / 4), ls=":", color="0.5", lw=1.2, label="reduced criterion")
ax.axhline(1 / 4, ls="--", color=RED, lw=1.0, label="chance")
ax.set_xticks(x); ax.set_xticklabels([k.replace(" ", "\n") for k in keys], fontsize=8)
ax.set_ylabel("final reward rate"); ax.set_ylim(0, 1.08); ax.set_title(f"Remedy diagnostic [{src}]")
ax.legend(frameon=False, fontsize=8); _clean(ax); plt.show()
print("claim status: live-oriented" if RESULT_MODE == "live" else "claim status: full-sweep cache")

### Sequential Maze Learning
Evaluates the device trace's ability to cross spatial and sequential gaps in the T-maze delayed-reward setting.


In [ ]:
if RESULT_MODE == "live":
    r = run_sequential(seeds=4, episodes=300, taus=(2.0, 10.0, 20.0), delays=(2, 4, 8))
    src = "LIVE reduced: 4 seeds, 300 episodes"
else:
    r = _cache("exp6_sequential.npy"); src = "full-sweep cache"
crit, chance = r["crit"], r["chance"]
fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.0, 3.7))
for name, c, lab in [("device", GREEN, "device trace"), ("rstdp", INDIGO, "R-STDP"), ("eprop", GOLD, "e-prop"), ("no_trace", GREY, "no trace")]:
    if name in r["curves"]:
        axA.plot(_smooth(r["curves"][name], win=30), color=c, lw=1.6, label=lab)
axA.axhline(crit, ls=":", color=GREY, lw=1.0); axA.axhline(chance, ls="--", color=RED, lw=0.9)
axA.set_xlabel("episode window"); axA.set_ylabel("reward rate"); axA.set_ylim(0.25, 1.03)
axA.set_title("Sequential maze learning"); axA.legend(frameon=False, fontsize=8); _clean(axA)
taus = np.asarray(r["taus"], float); delays = np.asarray(r["delays"], float); grid = np.asarray(r["grid"], float)
for tau, y, c in zip(taus, grid, VIR):
    axB.plot(delays, y, marker="o", ms=4, lw=1.6, color=c, label=rf"$\tau={tau:g}$ s")
axB.axhline(crit, ls=":", color=GREY, lw=1.0); axB.axhline(chance, ls="--", color=RED, lw=0.9)
axB.set_xscale("log"); axB.set_xticks(delays); axB.set_xticklabels([f"{d:g}" for d in delays])
axB.set_xlabel("delay D (s)"); axB.set_ylabel("reward rate"); axB.set_ylim(0.25, 1.03)
axB.set_title("Sequential retention law"); axB.legend(frameon=False, fontsize=8); _clean(axB)
fig.suptitle(f"Sequential delayed-credit diagnostic [{src}]", fontsize=11); fig.tight_layout(); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("criteria:", r.get("criteria", {}))
# Full-scale regeneration:
# python -m mrl_trace.bandit --bandit --full
# python -m mrl_trace.maze --exp6 --full